In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm

In [2]:
def load_dataset(folder_path):
    
    folder = Path(folder_path)
    csv_path = folder / "_classes.csv"
    
    # Read the CSV
    df = pd.read_csv(csv_path)
    
    # The label columns are: score 0, score 1, score 1 , score 2, score 3, score 4, score 5: Perfect Score
    # We need to find which column has the 1 and map it to 0-5
    # Columns 2-8 are the score columns (index 1 is Unlabeled, indices 2-8 are scores 0-5)
    label_cols = df.columns[2:8]  # Get score 0 through score 4 (6 columns for scores 0-5)
    
    images = []
    labels = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Loading {folder.name}"):
        filename = row.iloc[0]  # First column is filename
        img_path = folder / filename
        
        if not img_path.exists():
            print(f"Warning: {img_path} not found, skipping")
            continue
        
        # Load image and convert to grayscale
        img = Image.open(img_path).convert('L')  # 'L' mode is grayscale
        img_array = np.array(img)
        if img_array.shape != (640, 640):
            continue
        images.append(img_array)
        
        # Find which score column has the 1 (get the label 0-5)
        # Skip column 0 (filename) and column 1 (Unlabeled)
        score_values = row.iloc[2:8].values  # Columns for scores 0-5
        label = np.argmax(score_values)
        labels.append(label)
    
    X = np.array(images)
    y = np.array(labels)
    
    return X, y

In [3]:
base = Path(".")

print("Loading training data...")
X_train, y_train = load_dataset(base / "train")

print("Loading validation data...")
X_val, y_val = load_dataset(base / "valid")

print("Loading test data...")
X_test, y_test = load_dataset(base / "test")

X_train = X_train[:, ::2, ::2]
X_val = X_val[:, ::2, ::2]
X_test = X_test[:, ::2, ::2]

print(f"\nDataset shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

print(f"\nLabel distribution:")
print(f"Train: {np.bincount(y_train, minlength=6)}")
print(f"Val: {np.bincount(y_val, minlength=6)}")
print(f"Test: {np.bincount(y_test, minlength=6)}")

Loading training data...


Loading train: 100%|█████████████████████████████████████████| 20602/20602 [00:34<00:00, 594.01it/s]


Loading validation data...


Loading valid: 100%|███████████████████████████████████████████| 2168/2168 [00:02<00:00, 848.23it/s]


Loading test data...


Loading test: 100%|████████████████████████████████████████████| 1412/1412 [00:01<00:00, 931.91it/s]



Dataset shapes:
X_train: (20602, 320, 320), y_train: (20602,)
X_val: (2167, 320, 320), y_val: (2167,)
X_test: (1412, 320, 320), y_test: (1412,)

Label distribution:
Train: [ 548  268  794 2669 6511 9812]
Val: [ 29  39  92 306 737 964]
Test: [ 36  28  76 208 471 593]


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors and add channel dimension
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)  # (N, 1, H, W)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
y_val_t = torch.LongTensor(y_val)
y_test_t = torch.LongTensor(y_test)

batch_size = 64

# DataLoaders
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size)

# CNN
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
    
    def forward(self, x):
        return self.fc(self.conv(x))

num_classes = len(torch.unique(y_train_t))
model = CNN(num_classes)
device = torch.device('cuda:5' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
for epoch in range(20):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
    
    # Validation accuracy
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            correct += (model(X_batch).argmax(1) == y_batch).sum().item()
            total += len(y_batch)
    print(f"Epoch {epoch+1}: Val Acc = {correct/total:.4f}")

# Test accuracy
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        correct += (model(X_batch).argmax(1) == y_batch).sum().item()
        total += len(y_batch)
print(f"\nTest Accuracy: {correct/total:.4f}")

Epoch 1: Val Acc = 0.4509
Epoch 2: Val Acc = 0.5159
Epoch 3: Val Acc = 0.5445
